# Eksperimen Preprocessing — Telco Customer Churn

**Nama:** [Nama Anda]  
**Email:** [Email Dicoding Anda]  
**Dataset:** [Telco Customer Churn — Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  

---

> Notebook ini merupakan eksperimen preprocessing yang seluruh logikanya identik dengan `automate_dio.py`.

## Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
print('Libraries loaded ✅')

## 1. Data Loading

Memuat dataset dari file CSV. Kolom `customerID` diperlakukan sebagai identifier (bukan fitur prediktif). Kolom `Churn` adalah target.

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Shape   : {df.shape}')
print(f'Kolom   : {df.columns.tolist()}')
df.head()

## 2. Exploratory Data Analysis (EDA)

Memahami struktur data, tipe kolom, statistik deskriptif, dan distribusi fitur.

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Distribusi target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#4CAF50','#F44336'], startangle=90)
axes[0].set_title('Proporsi Churn')

sns.countplot(x='Churn', data=df, palette=['#4CAF50','#F44336'], ax=axes[1])
axes[1].set_title('Count Churn')

plt.suptitle('Distribusi Target Variable — Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(churn_counts)
print(f'Churn rate: {churn_counts["Yes"]/len(df)*100:.2f}%')

In [ ]:
# Distribusi fitur numerik
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df_temp  = df.copy()
df_temp['TotalCharges'] = pd.to_numeric(df_temp['TotalCharges'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(num_cols):
    df_temp[col].hist(bins=30, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(f'Distribusi {col}')
    axes[i].set_xlabel(col)
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur kategorikal vs Churn
cat_cols = ['gender','SeniorCitizen','Partner','Dependents',
            'Contract','PaymentMethod','InternetService','PhoneService']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(cat_cols):
    ax = axes[i//4][i%4]
    churn_pct = df.groupby(col)['Churn'].apply(lambda x: (x=='Yes').mean() * 100)
    churn_pct.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{col} vs Churn Rate (%)')
    ax.set_ylabel('Churn %')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 3. Missing Value Analysis

`TotalCharges` mengandung spasi kosong yang tampak seperti string, bukan angka. Perlu dikonversi ke numerik terlebih dahulu sebelum imputasi.

In [ ]:
print('Missing sebelum konversi:')
print(df.isnull().sum())

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing = df.isnull().sum()
print('Missing setelah konversi TotalCharges:')
print(missing[missing > 0])

In [ ]:
# Imputasi dengan median
median_val = df['TotalCharges'].median()
df['TotalCharges'].fillna(median_val, inplace=True)
print(f'TotalCharges diimputasi dengan median = {median_val:.2f}')
print(f'Total missing values tersisa: {df.isnull().sum().sum()}')

## 4. Duplicate Analysis

Memeriksa baris yang sepenuhnya identik dan menghapusnya jika ada.

In [ ]:
dup = df.duplicated().sum()
print(f'Jumlah duplikat: {dup}')

if dup > 0:
    df.drop_duplicates(inplace=True)
    print(f'  → {dup} baris dihapus. Shape baru: {df.shape}')
else:
    print('Tidak ada duplikat ✅')

print(f'Shape akhir: {df.shape}')

## 5. Outlier Analysis

Deteksi outlier menggunakan metode IQR. Penanganan: **Winsorization (capping)** — nilai di luar batas diganti dengan batas IQR.

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.7))
    axes[i].set_title(f'Boxplot — {col}')
plt.tight_layout()
plt.show()

In [ ]:
def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

for col in num_cols:
    low, high = iqr_bounds(df[col])
    n = df[(df[col] < low) | (df[col] > high)].shape[0]
    print(f'{col}: {n} outlier | lower={low:.2f}, upper={high:.2f}')

In [ ]:
# Capping
for col in num_cols:
    low, high = iqr_bounds(df[col])
    df[col] = df[col].clip(lower=low, upper=high)

print('Setelah capping:')
for col in num_cols:
    low, high = iqr_bounds(df[col])
    n = df[(df[col] < low) | (df[col] > high)].shape[0]
    print(f'  {col}: {n} outlier tersisa')

## 6. Feature Engineering

Membuat 4 fitur baru:
- `AvgMonthlyCharge` — rata-rata biaya per bulan  
- `NumServices` — jumlah layanan yang digunakan  
- `HasFamily` — memiliki partner atau tanggungan  
- `TenureGroup` — segmen masa berlangganan

In [ ]:
# 1. AvgMonthlyCharge
df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)

# 2. NumServices
service_cols = ['PhoneService','MultipleLines','InternetService',
                'OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']
no_vals = {'no','no internet service','no phone service'}

def count_services(row):
    return sum(1 for c in service_cols if str(row[c]).lower() not in no_vals)

df['NumServices'] = df.apply(count_services, axis=1)

# 3. HasFamily
df['HasFamily'] = ((df['Partner']=='Yes') | (df['Dependents']=='Yes')).astype(int)

# 4. TenureGroup
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                           labels=['0-12m','13-24m','25-48m','49-72m'], right=True)

print('Fitur baru:')
df[['AvgMonthlyCharge','NumServices','HasFamily','TenureGroup']].describe()

## 7. Encoding

- **LabelEncoder** → kolom biner (Yes/No, Male/Female)  
- **One-Hot Encoding** → kolom multi-kategori  
- Kolom `customerID` dihapus sebagai identifier non-prediktif

In [ ]:
df_enc = df.copy()
df_enc.drop(columns=['customerID'], inplace=True)

# Target
df_enc['Churn'] = df_enc['Churn'].map({'Yes': 1, 'No': 0})

# Label Encoding
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling']
le = LabelEncoder()
for col in binary_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    print(f'  LabelEncoded: {col}')

# One-Hot Encoding
ohe_cols = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
            'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
            'Contract','PaymentMethod','TenureGroup']
df_enc = pd.get_dummies(df_enc, columns=ohe_cols, drop_first=True)

# Bool → int
bool_c = df_enc.select_dtypes(include='bool').columns
df_enc[bool_c] = df_enc[bool_c].astype(int)

print(f'\nShape setelah encoding: {df_enc.shape}')
df_enc.head()

## 8. Scaling

Normalisasi fitur numerik menggunakan **StandardScaler** (mean=0, std=1).

In [ ]:
scale_cols = ['tenure','MonthlyCharges','TotalCharges','AvgMonthlyCharge','NumServices']

scaler = StandardScaler()
df_enc[scale_cols] = scaler.fit_transform(df_enc[scale_cols])

print('Setelah StandardScaler:')
df_enc[scale_cols].describe().round(4)

## 9. Train Test Split

Membagi data menjadi 80% train dan 20% test dengan **stratified sampling** untuk menjaga proporsi kelas.

In [ ]:
X = df_enc.drop(columns=['Churn'])
y = df_enc['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Total  : {len(df_enc)}')
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'\ny_train:\n{y_train.value_counts(normalize=True).round(3)}')
print(f'\ny_test:\n{y_test.value_counts(normalize=True).round(3)}')

## 10. Menyimpan Dataset Hasil Preprocessing

Dataset disimpan sebagai `telco_churn_preprocessed.csv` — siap digunakan oleh `modelling.py`.

In [ ]:
df_enc.to_csv('telco_churn_preprocessed.csv', index=False)

print('✅ Dataset berhasil disimpan: telco_churn_preprocessed.csv')
print(f'   Shape final : {df_enc.shape}')
print(f'   Jumlah fitur: {X_train.shape[1]}')
print(f'   Target      : Churn (0=No, 1=Yes)')